# Notebook 05 — Basic Neural Net with Embedding Layer (Section 5)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings, pickle
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, f1_score

import nltk
nltk.download('punkt', quiet=True)
from nltk.tokenize import word_tokenize
from collections import Counter
from tqdm import tqdm

OUTPUTS_DIR = Path("outputs")
MODELS_DIR = Path("models")
LOGS_DIR = Path("logs")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cpu


## 1. Load Data

In [2]:
train_df = pd.read_csv(OUTPUTS_DIR / 'train.csv').dropna(subset=['text', 'sentiment', 'star_rating', 'category'])
test_df = pd.read_csv(OUTPUTS_DIR / 'test.csv').dropna(subset=['text', 'sentiment', 'star_rating', 'category'])
print(f"Train: {len(train_df)} | Test: {len(test_df)}")

Train: 19282 | Test: 4821


## 2. Build Vocabulary

In [3]:
MAX_VOCAB = 20000
MAX_SEQ_LEN = 100
PAD_TOKEN = '<PAD>'
UNK_TOKEN = '<UNK>'

def tokenize(text):
    return word_tokenize(str(text).lower())

all_tokens = []
for text in tqdm(train_df['text'], desc="Tokenizing"):
    all_tokens.extend(tokenize(text))

counter = Counter(all_tokens)
vocab_words = [PAD_TOKEN, UNK_TOKEN] + [w for w, _ in counter.most_common(MAX_VOCAB - 2)]
word2idx = {w: i for i, w in enumerate(vocab_words)}

print(f"Vocabulary size: {len(word2idx)}")

def encode(text, max_len=MAX_SEQ_LEN):
    tokens = tokenize(text)[:max_len]
    ids = [word2idx.get(t, word2idx[UNK_TOKEN]) for t in tokens]
    ids += [word2idx[PAD_TOKEN]] * (max_len - len(ids))
    return ids

Tokenizing: 100%|██████████| 19282/19282 [00:02<00:00, 7899.80it/s]

Vocabulary size: 20000


## 3. Dataset Classes

In [4]:
class ReviewDataset(Dataset):
    def __init__(self, df, label_col, le=None):
        self.texts = [encode(t) for t in tqdm(df['text'], desc=f"Encoding {label_col}")]
        
        if le is None:
            self.le = LabelEncoder()
            self.labels = self.le.fit_transform(df[label_col].astype(str))
        else:
            self.le = le
            self.labels = le.transform(df[label_col].astype(str))
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        return torch.tensor(self.texts[idx], dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)
    
    def num_classes(self):
        return len(self.le.classes_)

## 4. Model Architecture

In [5]:
class TextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes, padding_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(embed_dim, 128)
        self.fc2 = nn.Linear(128, num_classes)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        emb = self.embedding(x)            # (B, L, E)
        pooled = emb.mean(dim=1)           # Global average pooling
        out = self.dropout(pooled)
        out = self.relu(self.fc1(out))
        out = self.dropout(out)
        return self.fc2(out)

## 5. Train Function

In [6]:
def train_model(task_name, label_col, num_epochs=10, batch_size=64, lr=1e-3, embed_dim=100):
    print(f"\n{'='*60}\nTraining task: {task_name}\n{'='*60}")
    
    train_ds = ReviewDataset(train_df, label_col)
    test_ds = ReviewDataset(test_df, label_col, le=train_ds.le)
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size)
    
    model = TextClassifier(len(word2idx), embed_dim, train_ds.num_classes()).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    writer = SummaryWriter(log_dir=str(LOGS_DIR / f'nn_embed_{task_name}'))
    
    best_f1 = 0
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        for texts, labels in train_loader:
            texts, labels = texts.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            logits = model(texts)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        # Eval
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for texts, labels in test_loader:
                texts = texts.to(DEVICE)
                preds = model(texts).argmax(dim=1).cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(labels.numpy())
        
        acc = accuracy_score(all_labels, all_preds)
        f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
        avg_loss = total_loss / len(train_loader)
        
        writer.add_scalar('Loss/train', avg_loss, epoch)
        writer.add_scalar('Accuracy/test', acc, epoch)
        writer.add_scalar('F1_macro/test', f1, epoch)
        
        print(f"Epoch {epoch+1}/{num_epochs} | Loss: {avg_loss:.4f} | Acc: {acc:.4f} | F1: {f1:.4f}")
        
        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), MODELS_DIR / f'nn_embed_{task_name}.pt')
    
    # Export embeddings to TensorBoard
    embed_weights = model.embedding.weight.data.cpu()[:500]
    labels_tb = [vocab_words[i] for i in range(500)]
    writer.add_embedding(embed_weights, metadata=labels_tb, tag=f'embedding_{task_name}')
    writer.close()
    
    print(f"Best F1: {best_f1:.4f} — saved to models/nn_embed_{task_name}.pt")
    
    # Final report
    model.load_state_dict(torch.load(MODELS_DIR / f'nn_embed_{task_name}.pt'))
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for texts, labels in test_loader:
            texts = texts.to(DEVICE)
            preds = model(texts).argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    
    print(classification_report(all_labels, all_preds, target_names=train_ds.le.classes_, zero_division=0))
    
    return {'task': task_name, 'accuracy': accuracy_score(all_labels, all_preds),
            'f1_macro': f1_score(all_labels, all_preds, average='macro', zero_division=0)}

## 6. Train All 3 Tasks

In [7]:
all_results = []
all_results.append(train_model('sentiment', 'sentiment', num_epochs=10))
all_results.append(train_model('star_rating', 'star_rating', num_epochs=10))
all_results.append(train_model('category', 'category', num_epochs=10))

results_df = pd.DataFrame(all_results)
print(results_df)
results_df.to_csv(OUTPUTS_DIR / 'results_nn_embed.csv', index=False)
print("Saved results to outputs/results_nn_embed.csv")
print("TensorBoard: tensorboard --logdir=logs/")


Training task: sentiment


Encoding sentiment: 100%|██████████| 4821/4821 [00:00<00:00, 7908.26it/s]


Epoch 1/10 | Loss: 0.7721 | Acc: 0.7681 | F1: 0.5504
Epoch 2/10 | Loss: 0.6163 | Acc: 0.7893 | F1: 0.5653
Epoch 3/10 | Loss: 0.5729 | Acc: 0.7951 | F1: 0.5735
Epoch 4/10 | Loss: 0.5491 | Acc: 0.7990 | F1: 0.5733
Epoch 5/10 | Loss: 0.5271 | Acc: 0.8023 | F1: 0.5843
Epoch 6/10 | Loss: 0.5132 | Acc: 0.8044 | F1: 0.5948
Epoch 7/10 | Loss: 0.4995 | Acc: 0.8075 | F1: 0.6075
Epoch 8/10 | Loss: 0.4898 | Acc: 0.8090 | F1: 0.6133
Epoch 9/10 | Loss: 0.4805 | Acc: 0.8069 | F1: 0.6152
Epoch 10/10 | Loss: 0.4677 | Acc: 0.8071 | F1: 0.6083
Best F1: 0.6152 — saved to models/nn_embed_sentiment.pt
              precision    recall  f1-score   support

    negative       0.83      0.94      0.88      2198
     neutral       0.36      0.06      0.11       676
    positive       0.80      0.91      0.85      1947

    accuracy                           0.81      4821
   macro avg       0.67      0.64      0.62      4821
weighted avg       0.75      0.81      0.76      4821


Training task: star_rating


Encoding star_rating: 100%|██████████| 4821/4821 [00:00<00:00, 7826.97it/s]


Epoch 1/10 | Loss: 1.3482 | Acc: 0.4850 | F1: 0.3191
Epoch 2/10 | Loss: 1.1717 | Acc: 0.5053 | F1: 0.3469
Epoch 3/10 | Loss: 1.1222 | Acc: 0.5186 | F1: 0.3757
Epoch 4/10 | Loss: 1.0938 | Acc: 0.5246 | F1: 0.3905
Epoch 5/10 | Loss: 1.0680 | Acc: 0.5229 | F1: 0.4013
Epoch 6/10 | Loss: 1.0495 | Acc: 0.5287 | F1: 0.4023
Epoch 7/10 | Loss: 1.0338 | Acc: 0.5291 | F1: 0.4157
Epoch 8/10 | Loss: 1.0170 | Acc: 0.5283 | F1: 0.4156
Epoch 9/10 | Loss: 1.0008 | Acc: 0.5267 | F1: 0.4329
Epoch 10/10 | Loss: 0.9835 | Acc: 0.5308 | F1: 0.4351
Best F1: 0.4351 — saved to models/nn_embed_star_rating.pt
              precision    recall  f1-score   support

           1       0.60      0.88      0.71      1462
           2       0.32      0.11      0.16       736
           3       0.34      0.17      0.23       676
           4       0.46      0.49      0.48      1012
           5       0.58      0.61      0.60       935

    accuracy                           0.53      4821
   macro avg       0.46      0.

Encoding category: 100%|██████████| 4821/4821 [00:00<00:00, 7654.68it/s]


Epoch 1/10 | Loss: 1.4566 | Acc: 0.4563 | F1: 0.2043
Epoch 2/10 | Loss: 1.2528 | Acc: 0.5468 | F1: 0.2835
Epoch 3/10 | Loss: 1.1539 | Acc: 0.5812 | F1: 0.3146
Epoch 4/10 | Loss: 1.0961 | Acc: 0.5961 | F1: 0.3387
Epoch 5/10 | Loss: 1.0533 | Acc: 0.6073 | F1: 0.3560
Epoch 6/10 | Loss: 1.0298 | Acc: 0.6148 | F1: 0.3679
Epoch 7/10 | Loss: 0.9991 | Acc: 0.6150 | F1: 0.3864
Epoch 8/10 | Loss: 0.9696 | Acc: 0.6212 | F1: 0.3996
Epoch 9/10 | Loss: 0.9493 | Acc: 0.6246 | F1: 0.3962
Epoch 10/10 | Loss: 0.9328 | Acc: 0.6239 | F1: 0.3993
Best F1: 0.3996 — saved to models/nn_embed_category.pt
                   precision    recall  f1-score   support

     Cancellation       0.26      0.12      0.16       281
Claims Processing       0.25      0.22      0.23       296
         Coverage       0.60      0.65      0.62      1225
 Customer Service       0.60      0.56      0.58       872
       Enrollment       0.36      0.03      0.05       146
          Pricing       0.70      0.81      0.75      2001


## Summary
- PyTorch: Embedding → GlobalAvgPool → Linear (2 layers)
- Trained on 3 tasks, logged to TensorBoard (loss, accuracy, F1, embeddings)